# 06 · Error analysis and export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/06_report.ipynb)

Show an item the model got wrong, and say whose fault it was.

```
  01_build_pool_<track>  →  02_sample  →  02b_add_samples  →  03_annotate  →  04_develop  →  05_test  →▶ 06_report
```

| | |
|---|---|
| **Reads** | the dev/test split (03) · the rounds and notes (04) · the frozen predictions and test log (05) |
| **Writes** | `outputs/` — the predictions CSV and a copy of your test set; and the numbers on screen that you write your report from |

---

This is the highest-value part of the whole project, and the one the Q&A will definitely go to. A low F1 with a clear account of *why* is worth more than a high one without.

Everything scored here is the **held-out test set**. The per-round table from notebooks 04 and 05 are your dev trail — how you got to the prompt — and the two are different claims.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, TEST_PATH,
                    DISAGREED_PATH, ADJUDICATED_PATH, PRED_PATH, ROUNDS_PATH,
                    NOTES_PATH, TESTLOG_PATH, PROMPT_FILE, SHEET_PATH,
                    describe)

# Loading files is plumbing.
from pipeline import (load_gold, load_predictions, load_json, save_json,
                      export_results, read_test_log, label_set)

# The scoring comes from scikit-learn, by its own names. You built your own
# versions of these on Day 2 S6 and checked them against these very functions;
# WHICH of them you report is the decision this notebook asks you to make.
from sklearn.metrics import (classification_report, cohen_kappa_score,
                             confusion_matrix, f1_score)

# The tables: your errors, which labels the model swaps, and the join against
# your coders' arguments. All met before — none of them calls the model.
import pandas as pd
from metrics import (show_errors, confused_pairs, errors_on_disagreed,
                     labels_of)
from pipeline import plot_confusion_matrix

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## Step 1 — Open the frozen run

Now we load the four files notebooks 04 and 05 left behind: the held-out test set, the dev half (only so you can say in the report how big it was), the predictions file, and the per-round table.

**Nothing in this notebook calls the model.** If a number here differs from what notebook 05 printed, you are loading a different file — not watching the model change its mind.

The last line prints three counts, and the test count and the prediction count **must be the same**. More predictions than test items means the run you froze was a run on dev — go back to step 2 of notebook 05.

Loading a frozen predictions file is what you did on Day 2 S6.

In [ ]:
# ══ STEP 1 · Load the frozen run ══════════════════════════════════════════
# Opens the held-out test set, the dev half, the predictions notebook 05 froze,
# and the per-round table.
# Creates: test, dev, pred_final, f1_by_round, NOTES, LABELS

test = load_gold(TEST_PATH)     # the held-out half — everything below is this
dev  = load_gold(DEV_PATH)      # only so the report can say how big it was
pred_final = load_predictions(PRED_PATH)
f1_by_round = load_json(ROUNDS_PATH, what="rounds",
                        made_by="notebook 05_test")
NOTES = load_json(NOTES_PATH, what="round notes",
                  made_by="notebook 04_develop")

# The label list every score below is computed over, worked out once here.
# config.yaml only sets labels_order when your labels are a SCALE, so for
# most schemes it is empty and the list comes off your gold set instead.
LABELS = LABELS_ORDER
if not LABELS:
    LABELS = label_set(test)

print(len(dev), "dev ·", len(test), "test ·", len(pred_final), "predictions")
print("scoring over:", LABELS)


### Your rounds, with the reason for each

Now we put the two tables side by side: what each round scored, and why you made that change. **This is your prompt-iterations section**, and it is the one section you cannot reconstruct afterwards — a stack of F1 numbers with no reasons attached is a list of things that happened, not an account of what you did.

A round whose reason still ends in `what happened: …` is one you have not finished. Go back to `04_develop.ipynb` and fill it in; the cell there saves it again.

In [ ]:
for name in f1_by_round:
    print(round(f1_by_round[name], 3), " ", name)
    print("        ", NOTES.get(name, "— no reason recorded —"))

### Your test log

How many times the held-out set was scored. **One row is what we expect.** More than one is allowed — that is why this file exists rather than a lock — but whichever row your headline comes from, your limitations section has to account for the others.

Look at `prompt_sha1`. Two rows with the *same* fingerprint are the same prompt run twice, which tells you something useful about how much the model's answers vary on their own. Two rows with *different* fingerprints are a prompt that changed after you had seen the held-out set — a different thing entirely, and one you have to say out loud.

In [ ]:
read_test_log(TESTLOG_PATH)


## What you have to work with

Everything below is assembled from these. Nothing here calls the model — all of it takes lists you have already loaded, which is what "frozen" means: from here on your numbers can only change if you load a different file.

| Call | What it gives you | First run |
|---|---|---|
| `labels_of(test)` | the gold labels as a plain list, ready for scoring | 06 |
| `classification_report(y, p, labels=LABELS)` | precision, recall and F1 for **every class** | Day 2 S6 |
| `f1_score(y, p, average="macro", labels=LABELS)` | one number: every class counts the same | Day 2 S6 |
| `f1_score(y, p, average="micro", labels=LABELS)` | one number: every **item** counts the same |Day 2 S6 |
| `f1_score(y, p, average="weighted", labels=LABELS)` | one number, classes weighted by how common they are | Day 2 S6 |
| `cohen_kappa_score(y, p)` | agreement with your gold, corrected for chance | Day 2 S6 |
| `cohen_kappa_score(y, p, weights="quadratic")` | the same, counting a near miss as a smaller error | Day 2 S6 |
| `confusion_matrix(y, p, labels=LABELS)` | which classes it confuses with which | Day 2 S6 |
| `plot_confusion_matrix(m, LABELS, title)` | that matrix, drawn | Day 2 S6 |
| `show_errors(test, pred)` | just the rows it got wrong | Day 3 |
| `errors_on_disagreed(errors, disagreed)` | the errors that land where YOUR coders also disagreed | 06 |
| `confused_pairs(errors)` | which label swaps the model made most often | 06 |

**The three averages are three different questions, and they disagree.** Macro asks how well you do on the average *class*, so a rare class counts as much as a common one — which is why it goes with a balanced sample. Micro asks how well you do on the average *item*, so the common classes dominate. Weighted sits between them. Run all three if you like; **which one you report is a decision, and `PLAN.md` §9 should already say which.** Choosing after you have seen all three is choosing the flattering one.

**Careful with the κ.** In notebook 03 it compared two **annotators**. Here it compares your gold labels against a **model**. Same kind of number, a different claim — do not swap them in the report.

## Step 2 — The headline number

**This one is the result.** It is measured on items your prompt was never tuned against, which is what makes it worth quoting; the per-round table above is the *story of how you got here*, and belongs in your prompt-iterations section rather than your evaluation section.

Compare it to your best dev round. If test came out lower, that is the ordinary outcome and the gap is itself a finding — roughly, how much of your improvement was tuning to those particular dev items rather than to the task. Say what you make of it in one sentence. And keep the sample size in view while you do: a few points either way on twenty-odd items is noise, so read a small gap as "we cannot tell these apart" rather than as a result.

Read the per-class table, not just the headline. *"Which class is it worst at, and what does it confuse that class with"* is a more useful sentence in a report than *"F1 = .62"*, and it is the one that says something about your scheme rather than about the model alone.

**Careful with the κ.** The one you computed in notebook 03 compared two **annotators**. This compares your gold labels against a **model**. Same kind of number, different claim — do not swap them in the report.

In [ ]:
# ══ STEP 2 · Score the frozen run ═════════════════════════════════════════
# Lines the gold labels up against the frozen predictions, then reports the
# per-class table. These are the numbers for your evaluation section.
# Creates: y_true, y_pred, macro_f1

# ✏️ Which one number you lead with, and whether a weighted κ belongs here.
#    Both follow from your label set, and PLAN.md §9 should already say.

y_true = labels_of(test)      # the gold side, as a plain list
y_pred = pred_final           # the model's side, already a plain list

# Every class, so you can say WHICH one it is worst at. That sentence is
# worth more in a report than the single number under it.
print(classification_report(y_true, y_pred, labels=LABELS,
                            zero_division=0))

# The headline. macro = every class counts the same, which is what a
# balanced sample was drawn for. Add the other averages if you want to see
# them; report the one you committed to.
macro_f1 = f1_score(y_true, y_pred, average="macro", labels=LABELS,
                    zero_division=0)
print("macro-F1 on the held-out test set:", round(macro_f1, 3))


### Now the rest of what you chose to report

Chance-corrected agreement with your gold, and the matrix. **Add the weighted κ only if your labels are a scale** — and if you do, `LABELS` in `config.yaml` has to be in scale order, or the weighting is computed over alphabetical order and the number means nothing.

Keep whatever you decide the same as notebooks 04 and 05 used, or the held-out row stops matching the table you put it at the bottom of.

In [ ]:
print("Cohen's kappa:", round(cohen_kappa_score(y_true, y_pred), 3))

# Labels on a scale? Then this one too — a near miss counts as a smaller error.
# print("weighted kappa:", round(cohen_kappa_score(
#     y_true, y_pred, labels=LABELS, weights="quadratic"), 3))

matrix = confusion_matrix(y_true, y_pred, labels=LABELS)
plot_confusion_matrix(matrix, LABELS, "Gold vs model, held-out set")

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We report ___ as our headline number, because ___.
>
> We did not lead with ___ because ___.
>
> The model was worst on ___, which it confused with ___.
>
> We read that as ___ about our scheme.

The second sentence is the one that shows you chose rather than accepted a default, and you can write it from `PLAN.md` §9 before the numbers exist. The last two come off the per-class table and the matrix, and they say something about your labels that the headline number cannot.

On a sample this size, a few points either way is noise. "We cannot tell these apart" is an honest sentence and it scores better than a difference you cannot support.

---

# Step 3 — Error analysis

**This is the part of the project the Q&A will actually go to.** A low F1 with a clear account of *why* beats a high one without, every time — and the account is only available to you because you built the gold set yourselves.

`show_errors` gives you every item the model got wrong. The question to ask of that table is the Day 2 S6 one: **is this the model's fault, or the scheme's?**

You are one of the few people who can answer it, because you built the gold set yourselves and you know which items you argued about. An item both your coders labelled at once and the model still missed is the model's. An item your coders split on is a boundary your scheme does not settle, and the model splitting on it too is evidence rather than coincidence.

So this step works the same way notebook 03 step 2 did — find the label pair that keeps swapping, then say what is true of that boundary.

In [ ]:
# ══ STEP 3 · The errors ═══════════════════════════════════════════════════
# Lists every test item the model got wrong, and shows you the first fifteen.
# Creates: errors

errors = show_errors(test, pred_final)
errors.head(15)


### Now look at one label's misses at a time

`errors` is a table, and `errors[errors.gold == "…"]` keeps only the rows whose `gold` column equals what you put in the quotes. Start with the class the per-class table scored worst.

The cell picks that class for you, but the choice worth making is *which* label — replace the first line with e.g. `LOOK_AT = "Move 2"` and re-run it as often as you like.

In [ ]:
# The label with the most misses. Counting them out, one row at a time.
miss_counts = {}
for label in errors["gold"]:
    if label not in miss_counts:
        miss_counts[label] = 0
    miss_counts[label] = miss_counts[label] + 1

LOOK_AT = max(miss_counts, key=miss_counts.get)   # <- or type a label yourself
print("looking at:", LOOK_AT, "·", miss_counts)

errors[errors["gold"] == LOOK_AT]

### Now the cross-reference — where did YOUR coders disagree?

Notebook 03 already worked this out and saved it, so this is one file being opened rather than a second trip to the Google Sheet. It works whether or not the sheet still exists, and it cannot disagree with 03 about who the coders were.

**If it says the file is missing**, notebook 03 was run before this file was part of the project. Re-run its step 3 — step 4 below asks whether your coders and the model split on the same items, and without this file there is nothing to answer it with.

In [ ]:
disagreed = load_json(DISAGREED_PATH)   # written by notebook 03, step 3
print(len(disagreed), "rows your coders labelled differently")

### Now the number this whole project has been building towards

How many of the model's errors land on the very items your two coders could not agree on either. If they cluster there, what you have measured is a fuzzy boundary in your annotation scheme rather than a stupid model — and that is a better finding than a clean F1.

Write it down **either way**: a low overlap is just as reportable, and means something else. Note too that `disagreed` covers the whole sheet while `errors` covers only the test half, so the overlap is smaller than it would have been without a split. Nothing is broken — the split kept every item's original id precisely so this join still lines up.

In [ ]:
overlap = errors_on_disagreed(errors, disagreed)
print("ids to read again before you blame the model:", overlap)

## Step 4 — Which boundary is this, and is it the same one?

Two questions, and the notebook computes the numbers for both. What it cannot do is say what they mean, and that is the last genuinely analytical thing in the project. Do it **together, out loud, reading the actual sentences** — it takes about fifteen minutes and it is the single hardest thing to reconstruct a week later.

**First: which two labels does the model confuse most often?** `confused_pairs` counts the `gold -> pred` swaps, commonest first. This is the same reading you made of the coder-vs-coder matrix in notebook 03 step 2, made now of the model.

**Then: is it the same pair your own coders disagreed about?** `overlap` from step 3 already has the ids. If the two answers point at the same boundary, you have something worth saying: two people who read the guidelines and one model that did not all failed at the same place, so the problem is in what the scheme says, not in who or what was reading it. That is a better finding than a clean F1, and nobody who did not build their own gold set can report it.

If they point at different boundaries, that is reportable too, and it means something else — the model is failing somewhere your coders found easy.

Read the sentences behind the commonest pair before you write anything. The count tells you where to look; it does not tell you what is there.

In [ ]:
# ══ STEP 4 · The boundary the model gets wrong ════════════════════════════
# Ranks the label swaps the model made, then says how many of its errors land
# on the items your own coders argued about.
# Creates: pairs

pairs = confused_pairs(errors)

# How much of the error set sits on items your coders split on too.
print(len(overlap), "of", len(errors), "errors are on rows you argued about")

pairs


### Now read the items behind that pair

The commonest swap, as sentences. Change the two labels to look at any other pair in the table above — the one that surprises you is often worth more than the one that is biggest.

In [ ]:
# The commonest swap, from the table above. Or type your own two labels.
GOLD_IS, PRED_WAS = pairs.iloc[0]["gold"], pairs.iloc[0]["pred"]

print("items labelled", GOLD_IS, "that the model called", PRED_WAS)
errors[(errors["gold"] == GOLD_IS) & (errors["pred"] == PRED_WAS)]

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> The model most often labelled ___ items as ___, ___ times out of ___ errors.
>
> ___ of its errors fell on rows our own coders had disagreed about.
>
> We read that as ___, and our scheme would have to ___ to settle those items.
>
> With another week we would ___, because ___.

The second sentence is the one this whole project was built to let you write. Your coders' disagreements are independent evidence that a boundary is unclear — evidence collected before anyone saw what the model would do with it.

The third is where the two readings meet: if notebook 03 named the same pair, say so explicitly. If it named a different one, say that instead — it is just as real a result, and pretending otherwise is visible in the Q&A.

## Step 5 — Export

Writes two files, both stamped with your group name: your test set, and a per-item predictions CSV with one row per item — id, gold label, predicted label, whether they matched, and the text.

These are what make your reported F1 checkable by someone else: which items it was measured on, and what the model said about each one. The CSV is also what you sort to find the misses for your error analysis.

With `dev=dev`, the saved items are named `_test` rather than `_gold`, because with a split the half you scored is the test half and calling it "gold" would misdescribe it.

**The report itself you write, in Word.** Nothing here drafts it for you. Everything it needs is on screen in this notebook: the rounds table in step 1, the per-class scores and confusion matrix in step 3, and the errors in step 4.

In [ ]:
# ══ STEP 5 · Export ═══════════════════════════════════════════════════════
# Writes two files into `../outputs/`: your test set, and the per-item
# predictions CSV.

export_results(TRACK, test, pred_final, OUT_DIR,
               group=GROUP, run=RUN, dev=dev)


---

## Hand it in

One command collects everything into a folder next to the repo, keeping the `scripts/ · prompts/ · data/ · notebooks/ · outputs/` layout — because that layout *is* the reproducibility checklist from S10, and because the notebooks' paths only resolve if it stays intact.

```bash
python scripts/make_submission.py --group groupA
```

It deliberately leaves out `.git/`, `.venv/`, your `.env` (**it holds your API key**), the big pools in `data/pools/`, and anything ICNALE-derived.

Then: find the folder in Drive → right-click → **Download** → upload the zip to the *Final mini-project* assignment in Google Classroom → **Turn in**. One zip per group, with every member's name in `PLAN.md`.

**Your two-page report is not in that zip.** You write it yourself, in Word, from the numbers this notebook printed above, and upload it to Classroom separately — one per person, not one per group.

Before you do, check that **all five notebooks run top to bottom on a fresh runtime**, in order. If they only work in the session where you built them piece by piece, they do not yet reproduce — and 02 through 05 handing files to each other is exactly what makes that checkable.

In [ ]:
# Optional: build the bundle from here instead of a terminal.
# !cd .. && python scripts/make_submission.py --group $GROUP
